# 00 — Environment check

Run this first on every machine (laptop and LinuxONE). The output doubles as **deployment evidence** for the judges:
architecture (`s390x` on LinuxONE), byte order (big-endian on IBM Z), library versions and CPU features.

In [ ]:
import importlib
import os
import platform
import sys
from pathlib import Path

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"Architecture : {platform.machine()}")
print(f"Byte order   : {sys.byteorder}")
print(f"CPU count    : {os.cpu_count()}")
print(f"On IBM Z     : {platform.machine() == 's390x'}")

In [ ]:
REQUIRED = ["numpy", "pandas", "sklearn", "xgboost", "networkx", "matplotlib", "joblib", "requests"]

missing = []
for name in REQUIRED:
    try:
        mod = importlib.import_module(name)
        print(f"{name:12s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        missing.append(name)
        print(f"{name:12s} MISSING")

if missing:
    print("\nInstall the missing packages before continuing (see README for s390x notes).")

On IBM Z, `/proc/cpuinfo` lists CPU facilities. `vx` is the vector facility; `nnpa` is the
Neural Network Processing Assist (the on-chip AI accelerator on z16 / LinuxONE 4 and newer).
Only claim accelerator use in the pitch if you actually route inference through it.

In [ ]:
cpuinfo = Path("/proc/cpuinfo")
if cpuinfo.exists():
    features = next((l.split(":", 1)[1].split() for l in cpuinfo.read_text().splitlines()
                     if l.startswith("features")), [])
    if features:
        print("CPU features:", " ".join(features))
        print("Vector facility (vx):", "vx" in features)
        print("AI accelerator (nnpa):", "nnpa" in features)
    else:
        print("No 'features' line (expected on x86/ARM; present on s390x).")